## Environment Setup and Library Initialization

This step initializes the development environment by importing the required Python libraries for data processing, Large Language Model (LLM) interaction, and graph database integration.

### Objective
The goal of this step is to prepare the environment for performing **LLM-based Named Entity Recognition (NER)** and **relationship extraction** from the dataset.

### Libraries Used

**1. Pandas**
- Used for loading and manipulating structured datasets.
- Enables efficient handling of tabular data such as CSV or Excel files.

**2. Transformers**
- Provides access to pretrained Large Language Models (LLMs).
- In this project, it will be used to load the **Mistral model** for entity and relationship extraction.

**3. Torch**
- PyTorch is the deep learning framework used to run transformer-based models.

**4. Neo4j Python Driver**
- Used to connect Python applications with the Neo4j graph database.
- Enables storing extracted entities and relationships as nodes and edges.

**5. NetworkX**
- A Python library for creating and visualizing knowledge graphs.

### Outcome
After executing this cell, the environment will be ready to load the dataset, interact with the Mistral LLM, and build the knowledge graph.

In [2]:
# Install required libraries (run once)
!pip install pandas transformers torch neo4j networkx openpyxl

# Import libraries
import pandas as pd
import torch
import json
import networkx as nx

from transformers import AutoTokenizer, AutoModelForCausalLM
from neo4j import GraphDatabase

Defaulting to user installation because normal site-packages is not writeable


## Dataset Loading and Initial Exploration

In this step, we load the preprocessed support ticket dataset that will be used for performing entity and relationship extraction using a Large Language Model (LLM).

### Objective

The objective of this stage is to read the cleaned dataset into a structured format so that it can be processed efficiently during the entity extraction phase.

The dataset used in this project consists of cleaned support tickets that contain textual descriptions of issues reported by users. These descriptions will serve as the input for the LLM-based Named Entity Recognition (NER) and relationship extraction pipeline.

### Process

1. Load the dataset from the Excel file `cleaned_tickets.xlsx`.
2. Store the dataset in a Pandas DataFrame.
3. Inspect the dataset structure to understand available columns and data distribution.



In [3]:
# Load the cleaned dataset

file_path = "cleaned_tickets.xlsx"

df = pd.read_excel(file_path)

# Display dataset shape
print("Dataset Shape:", df.shape)

# Display first few rows
df.head()

Dataset Shape: (8469, 21)


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,...,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating,Ticket State,Resolution Status,Severity,Resolution Time Category
0,1,marisa obrien,carrollallison@example.com,32,other,gopro hero,2021-03-22,technical issue,product setup,i'm having an issue with the {product_purchase...,...,not resolved yet,critical,social media,2023-06-01 12:15:36,not available,not rated,pending customer response,Unresolved,critical,Unknown
1,2,jessica rios,clarkeashley@example.com,42,female,lg smart tv,2021-05-22,technical issue,peripheral compatibility,i'm having an issue with the {product_purchase...,...,not resolved yet,critical,chat,2023-06-01 16:45:38,not available,not rated,pending customer response,Unresolved,critical,Unknown
2,3,christopher robbins,gonzalestracy@example.com,48,other,dell xps,2020-07-14,technical issue,network problem,i'm facing a problem with my {product_purchase...,...,case maybe show recently my computer follow.,low,social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3,closed,Resolved,low,Resolved
3,4,christina dillon,bradleyolson@example.org,27,female,microsoft office,2020-11-13,billing inquiry,account access,i'm having an issue with the {product_purchase...,...,try capital clearly never color toward story.,low,social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3,closed,Resolved,low,Resolved
4,5,alexander carroll,bradleymark@example.com,67,female,autodesk autocad,2020-02-04,billing inquiry,data loss,i'm having an issue with the {product_purchase...,...,west decision evidence bit.,low,email,2023-06-01 00:12:42,2023-06-01 19:53:42,1,closed,Resolved,low,Resolved


In [4]:
df.columns

Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating', 'Ticket State', 'Resolution Status',
       'Severity', 'Resolution Time Category'],
      dtype='object')

## Entity and Relationship Extraction using the Mistral LLM

After loading the dataset, the next step is to process each ticket description using the Mistral Large Language Model. The goal is to extract meaningful entities and relationships from the unstructured ticket text.

### Objective

The objective of this step is to convert ticket descriptions into structured knowledge that can later be used to build a knowledge graph.

### Process

1. Iterate through each ticket description in the dataset.
2. Send the ticket text to the Mistral model using Ollama.
3. Ask the model to extract entities and relationships.
4. Store the extracted information for further processing.

The extracted entities and relationships will later be used to construct a graph structure for visualization and analysis.

In [ ]:
import ollama

results = []

for ticket in df['Ticket Description'].dropna():

    prompt = f"""
    Extract entities and relationships from the following support ticket.

    Ticket:
    {ticket}

    Return the result in this format:

    Entities: []
    Relationships: []
    """

    response = ollama.chat(
        model="mistral",
        messages=[{"role": "user", "content": prompt}]
    )

    output = response['message']['content']

    results.append({
        "ticket": ticket,
        "extracted_info": output
    })

results

## Structuring Extracted Entities and Relationships

After extracting entities and relationships from ticket descriptions using the Mistral Large Language Model, the next step is to organize the results into a structured format.

### Objective

The objective of this step is to convert the raw responses generated by the language model into a structured tabular format that can be easily analyzed and processed.

### Process

1. Collect the extracted outputs generated by the Mistral model.
2. Combine the original ticket descriptions with the extracted information.
3. Store the results in a pandas DataFrame.

### Output

The resulting DataFrame contains:

- Ticket Description
- Extracted Entities
- Extracted Relationships

### Importance

Structuring the extracted information allows the data to be easily processed for downstream tasks such as relationship analysis, knowledge graph construction, and visualization.

In [ ]:
import pandas as pd

structured_df = pd.DataFrame(results)

structured_df.head()

In [ ]:
structured_df['Entities'] = structured_df['extracted_info'].str.extract(...)
structured_df['Relationships'] = structured_df['extracted_info'].str.extract(...)

## Knowledge Graph Construction

After extracting entities and relationships from the ticket descriptions, the next step is to construct a knowledge graph.

### Objective

The objective of this step is to represent the extracted information as a graph structure where:

- Entities are represented as **nodes**
- Relationships are represented as **edges**

### Process

1. Read the extracted entities and relationships from the structured dataset.
2. Create nodes representing entities.
3. Create edges representing relationships between entities.
4. Store the graph structure for visualization and further analysis.

### Importance

Knowledge graphs allow complex relationships between entities to be visualized and analyzed effectively. This representation helps in understanding patterns within support tickets and identifying common issues, dependencies, and system interactions.

In [ ]:
import networkx as nx

G = nx.Graph()

for _, row in structured_df.iterrows():
    
    entities = str(row['Entities']).split(',')
    
    for entity in entities:
        G.add_node(entity.strip())

    relationships = str(row['Relationships']).split(',')
    
    for rel in relationships:
        parts = rel.split("->")
        
        if len(parts) == 2:
            source = parts[0].strip()
            target = parts[1].strip()
            
            G.add_edge(source, target)

print("Graph created successfully")